In [67]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.dates as mdates
import numpy as np

import intake

import sys
sys.path.append("/Users/alessandrostorer/Desktop/PhD/1st_yy/work/codes/BCO_observational/")  # for great circle measurements
import functions_collector



# sys.path.append("/work/mh1498/m301248/TCO_data/COARE-algorithm/Python/COARE3.5/") 

# sys.path.append("/work/mh1498/m301248/TCO_data/BCO_observational/mass_fluxes_profiles/yk22_mass_fluxes.py")
# import yk22_mass_fluxes

import metpy

# import meteo


import pandas as pd
from IPython.display import display, HTML


## working local
local_path = {
              "joanne4": "/Users/alessandrostorer/Desktop/PhD/1st_yy/work/datasets/dropsondes_34/EUREC4A_JOANNE_Dropsonde-RD41_Level_4_v2.0.0.nc",
              "joanne3": "/Users/alessandrostorer/Desktop/PhD/1st_yy/work/datasets/dropsondes_34/EUREC4A_JOANNE_Dropsonde-RD41_Level_3_v2.0.0.nc",
              "beach4": "/Users/alessandrostorer/Desktop/PhD/1st_yy/work/datasets/dropsondes_34/ORCESTRA_BEACH_level_4.nc",
              "beach3":  "/Users/alessandrostorer/Desktop/PhD/1st_yy/work/datasets/dropsondes_34/ORCESTRA_BEACH_level_3.nc"
              }

In [68]:
## constants I always need
Lv = 2.5e6  # J/kg
cp = 1004.7 # J/kg K

Rd = 287.   # J / kg K

In [69]:
'''
beach.to_netcdf("/Users/alessandrostorer/Desktop/PhD/1st_yy/work/datasets/ORCESTRA_BEACH_level_4.nc")
beach3.to_netcdf("/Users/alessandrostorer/Desktop/PhD/1st_yy/work/datasets/ORCESTRA_BEACH_level_3.nc")
'''


## ipfs opening
# beach  = xr.open_dataset("ipfs://bafybeibgeeqs5uhmbqy4hz4v3pihrfisiklcetisxkf63d2r473szaprwi", engine="zarr")  # level 4
# beach3 = xr.open_dataset("ipfs://bafybeiczbv7mycr2jois6t4dq3zwiltycomwo5xxvjqcjz2ot3newzar6q", engine="zarr")  # level 3

## local access
joanne4  = xr.open_dataset(local_path["joanne4"])  # level 4
joanne3 = xr.open_dataset(local_path["joanne3"])  # level 3

halo3 = joanne3.platform_id == "HALO"
halo4 = joanne4.platform_id == "HALO"
joanne3 = joanne3.where(halo3, drop=True).sortby("launch_time")
joanne4 = joanne4.where(halo4, drop=True).sortby("circle_time")


In [70]:
joanne3


<xarray.Dataset> Size: 81MB
Dimensions:            (sonde_id: 810, alt: 1001, nv: 2)
Coordinates:
  * sonde_id           (sonde_id) <U13 42kB 'HALO-0119_s02' ... 'HALO-0218_s01'
    launch_time        (sonde_id) datetime64[ns] 6kB 2020-01-19T16:55:14 ... ...
  * alt                (alt) int16 2kB 0 10 20 30 40 ... 9970 9980 9990 10000
    interpolated_time  (sonde_id, alt) datetime64[ns] 6MB ...
    lat                (sonde_id, alt) float32 3MB ...
    lon                (sonde_id, alt) float32 3MB ...
Dimensions without coordinates: nv
Data variables: (12/23)
    p                  (sonde_id, alt) float32 3MB nan 1.013e+05 ... nan nan
    ta                 (sonde_id, alt) float32 3MB nan 299.0 298.9 ... nan nan
    rh                 (sonde_id, alt) float32 3MB nan 0.7669 0.7567 ... nan nan
    wspd               (sonde_id, alt) float32 3MB nan 9.681 ... 22.96 23.02
    wdir               (sonde_id, alt) float32 3MB nan 67.17 ... 326.2 326.3
    u                  (sonde_id, alt) float32 3MB nan -8.922 ... 12.78 12.77
    ...                 ...
    N_gps              (sonde_id, alt) float32 3MB nan 2.0 2.0 ... 1.0 1.0 1.0
    m_p                (sonde_id, alt) float32 3MB nan 2.0 2.0 ... 1.0 1.0 1.0
    m_ta               (sonde_id, alt) float32 3MB nan 2.0 2.0 ... 1.0 1.0 1.0
    m_rh               (sonde_id, alt) float32 3MB nan 2.0 2.0 ... 1.0 1.0 1.0
    m_gps              (sonde_id, alt) float32 3MB nan 2.0 2.0 ... 1.0 1.0 1.0
    alt_bnds           (alt, nv, sonde_id) float64 13MB -5.0 -5.0 ... 1e+04
Attributes: (12/16)
    title:                   EUREC4A JOANNE Level-3
    doi:                     10.25326/246
    created with:            run_joanne.py doi:10.5281/zenodo.5521192
    Conventions:             CF-1.8
    campaign_id:             EUREC4A
    project_id:              JOANNE
    ...                      ...
    JOANNE_version:          2.0.0
    author:                  Geet George
    author_email:            geet.george@mpimet.mpg.de
    featureType:             trajectory
    reference:               George et al., JOANNE : Joint dropsonde Observat...
    creation_time:           2021-09-22 14:35:24.817595 UTC

In [71]:
joanne4

<xarray.Dataset> Size: 9MB
Dimensions:          (circle: 71, alt: 1001, sounding: 13)
Coordinates:
  * circle           (circle) int64 568B 14 15 16 17 18 19 ... 79 80 81 82 83 84
    circle_lon       (circle) float32 284B -57.86 -57.7 -57.71 ... -57.67 -57.65
    circle_lat       (circle) float32 284B 13.27 13.27 13.28 ... 13.33 13.33
    circle_time      (circle) datetime64[ns] 568B 2020-01-19T17:53:57.800000 ...
    segment_id       (circle) <U12 3kB 'HALO-0119_c1' ... 'HALO-0215_c6'
  * alt              (alt) int16 2kB 0 10 20 30 40 ... 9960 9970 9980 9990 10000
  * sounding         (sounding) int64 104B 0 1 2 3 4 5 6 7 8 9 10 11 12
Data variables: (12/36)
    platform_id      (circle) object 568B 'HALO' 'HALO' 'HALO' ... 'HALO' 'HALO'
    flight_altitude  (circle) float32 284B 1.026e+04 1.029e+04 ... 1.119e+04
    circle_diameter  (circle) float32 284B 1.867e+05 2.225e+05 ... 2.236e+05
    u                (circle, alt) float32 284kB nan -8.829 ... 25.54 25.55
    dudx             (circle, alt) float32 284kB nan 8.21e-06 ... -6.027e-06
    dudy             (circle, alt) float32 284kB nan 6.836e-06 ... -1.797e-05
    ...               ...
    se_dtadx         (circle, alt) float32 284kB nan -0.0 ... -0.0 -0.0
    se_dtady         (circle, alt) float32 284kB nan -0.0 ... -0.0 -0.0
    se_D             (circle, alt) float32 284kB nan 6.747e-06 ... 8.735e-06
    se_vor           (circle, alt) float32 284kB nan 6.725e-06 ... 9.522e-06
    se_W             (circle, alt) float32 284kB 0.0 6.747e-06 ... 0.01219
    omega            (circle, alt) float32 284kB nan 0.002026 ... nan nan
Attributes: (12/16)
    title:                   EUREC4A JOANNE Level-4
    doi:                     10.25326/246
    created with:            run_joanne.py doi:10.5281/zenodo.5521192
    Conventions:             CF-1.8
    campaign_id:             EUREC4A
    project_id:              JOANNE
    ...                      ...
    JOANNE_version:          2.0.0
    author:                  Geet George
    author_email:            geet.george@mpimet.mpg.de
    featureType:             trajectory
    reference:               George et al., JOANNE : Joint dropsonde Observat...
    creation_time:           2021-09-22 15:58:46.773760 UTC

In [ ]:
sondes4 = joanne4.sonde_id
sondes_per_circle = 13 - (sondes4 == '').sum(dim="sounding")


sonde_sel = sondes4.values.flatten()
sonde_sel = sonde_sel[sonde_sel != '']
joanne3 = joanne3.sel(sonde_id=sonde_sel)

compute $qv$

In [ ]:
qv = joanne3.q * joanne3.v

In [ ]:
BCO_lon = -59.4288
BCO_lat = 13.1627

# - set control volume edges
vol_lon = [BCO_lon-0.5+360 , BCO_lon+15+360]  #[-60+360, -58+360]
vol_lat = [BCO_lat-5     , BCO_lat+5]  #[12.5, 14.5]


# - presenting area of interest

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

# Add base features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set_extent([vol_lon[0] - 1., vol_lon[1] + 30., vol_lat[0] - 8., vol_lat[1] + 5.], crs=ccrs.PlateCarree())


gl = ax.gridlines(draw_labels=True, 
                  linewidth=0.5, color='gray', alpha=0.5, linestyle='--')

# Turn on labels on bottom and left only
gl.top_labels = False
gl.right_labels = False

# Optionally format ticks (degrees + symbols)
gl.xformatter = cticker.LongitudeFormatter()
gl.yformatter = cticker.LatitudeFormatter()




## scatter the dropsondes lon and lat
sc = ax.scatter(joanne3.lon.isel(alt=2), joanne3.lat.isel(alt=2), c=qv.sel(alt=slice(50, 500)).mean(dim="alt"), cmap="BrBG", transform=ccrs.PlateCarree())
cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label(r"$\rm m~s^{-1}~kg~kg^{-1}$")
ax.set_title(r"$qv$ in each single dropsonde")


In [ ]:
np.cos(np.pi*10/180)

### assign circle labels

In [ ]:
## assign labels to define which circle each sonde belongs to

circle_labels = np.repeat(
    joanne4.circle.values,
    sondes_per_circle.values
)

circle_lon_rep = np.repeat(
    joanne4.circle_lon.values,
    sondes_per_circle.values
)

circle_lat_rep = np.repeat(
    joanne4.circle_lat.values,
    sondes_per_circle.values
)

joanne3_clean = joanne3.assign_coords(
    circle=("sonde_id", circle_labels), 
    circle_lat_rep=("sonde_id", circle_lat_rep), 
    circle_lon_rep=("sonde_id", circle_lon_rep)
)

circle_da = xr.DataArray(
    circle_labels,
    dims="sonde_id",
    coords={"sonde_id": joanne3_clean.sonde_id},
    name="circle"
)

In [ ]:
import importlib
importlib.reload(functions_collector)

In [ ]:
from functions_collector import great_circle, great_circle_haversine

def sign(x, y):
    return xr.where(x >= y, 1.0, -1.0).where(x.notnull() & y.notnull())



x = sign(joanne3_clean.lon, joanne3_clean.circle_lon_rep)*great_circle_haversine(joanne3_clean.lon, joanne3_clean.lat, joanne3_clean.circle_lon_rep, joanne3_clean.lat)
y = sign(joanne3_clean.lat, joanne3_clean.circle_lat_rep)*great_circle_haversine(joanne3_clean.lon, joanne3_clean.lat, joanne3_clean.lon, joanne3_clean.circle_lat_rep)

joanne3_clean['x'], joanne3_clean['y'] = x, y

In [ ]:
## from JOANNE's computing routine
## thanks Geet George!!
def fit2d(x, y, u):
    """
    estimate a 2D linear model to calculate u-values from x-y coordinates

    :param x: x coordinates of data points. shape: (...,M)
    :param y: y coordinates of data points. shape: (...,M)
    :param u: data values. shape: (...,M)

    all points along the M dimension are expected to belong to the same model
    all other dimensions are for different models

    :returns: intercept, dudx, dudy. all shapes: (...)
    """
    u_ = u
    # to fix nans, do a copy
    u = np.array(u, copy=True)
    # a does not need to be copied as this creates a copy already
    a = np.stack([np.ones_like(x), x, y], axis=-1)

    # for handling missing values, both u and a are set to 0, that way
    # these items don't influence the fit
    invalid = np.isnan(u) | np.isnan(x) | np.isnan(y)
    under_constraint = np.sum(~invalid, axis=-1) < 6
    u[invalid] = 0
    a[invalid] = 0

    a_inv = np.linalg.pinv(a)

    intercept, dudx, dudy = np.einsum("...rm,...m->r...", a_inv, u)

    intercept[under_constraint] = np.nan
    dudx[under_constraint] = np.nan
    dudy[under_constraint] = np.nan

    return intercept, dudx, dudy, u_


def fit2d_xr(x, y, u, input_core_dims, output_core_dims):
    # input and output dims must be a list

    return xr.apply_ufunc(
        fit2d,
        x,
        y,
        u,
        input_core_dims=[input_core_dims, input_core_dims, input_core_dims],
        output_core_dims=[(), (), (), output_core_dims],
    )


### local corrections
def fit2d_xr_loc(x, y, u):
    return xr.apply_ufunc(
        fit2d,
        x,
        y,
        u,
        input_core_dims=[
            ["sonde_id"],
            ["sonde_id"],
            ["sonde_id"],
        ],
        output_core_dims=[
            [],
            [],
            [],
            ["sonde_id"],
        ],
        vectorize=False,
    )


def fit_circle_loc(g, u_name):
    intercept, dudx, dudy, _ = fit2d_xr_loc(
        g.x,
        g.y,
        g[u_name],
    )

    grad_x_name = "d_"+u_name+"_dx"
    grad_y_name = "d_"+u_name+"_dy"


    ## compute standard errors here directly
    dx_mean = g.x.mean(dim="sonde_id").values
    dy_mean = g.y.mean(dim="sonde_id").values

    dx_denominator = np.sqrt(((g.x - dx_mean) ** 2).sum(dim="sonde_id"))
    dy_denominator = np.sqrt(((g.y - dy_mean) ** 2).sum(dim="sonde_id"))

    ## compute deviation of regressed estimate from observations
    u_err = g[u_name] - ( intercept + (dudx * g.x) + (dudy * g.y) )
    u_sq_sum = (u_err**2).sum(dim="sonde_id", skipna=True)    #np.nansum((u_err ** 2), axis=1)
    u_n = u_err.notnull().sum(dim="sonde_id")

    u_numerator = np.sqrt(u_sq_sum / (u_n - 3))
    
    se_dudx = u_numerator / dx_denominator
    se_dudy = u_numerator / dy_denominator

    var_name_dx = "se_d" + u_name + "dx"
    var_name_dy = "se_d" + u_name + "dy"

    return xr.Dataset({
        "area_mean": intercept,
        grad_x_name: dudx,
        grad_y_name: dudy,
        var_name_dx: se_dudx,
        var_name_dy: se_dudy
    })




In [ ]:
joanne3_clean["qv"] = (joanne3_clean.q * joanne3_clean.v)

result = joanne3_clean.groupby("circle").map(lambda g: fit_circle_loc(g, "qv"))

In [ ]:
joanne3_clean

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True )
fig.suptitle("relative errors on gradients using Geet's formulas")

ax[0].set_title(r"$\partial_x \phi$")
rel_err_dx = (100 * (result.se_dqvdx / np.abs(result.d_qv_dx))).sel(alt=slice(0, 1500)).groupby_bins("alt", bins=16).mean(dim="alt").to_pandas()

rel_err_dx.boxplot(
        ax=ax[0],
        vert=False,
        whis=(5, 95),
        patch_artist=True,
        boxprops=dict(
            facecolor="lightsteelblue",
            edgecolor="k"
        ),
        medianprops=dict(
            color="crimson",
            linewidth=2
        ),
        whiskerprops=dict(color="k"),
        capprops=dict(color="k"),
        flierprops=dict(
            marker="o",
            markersize=2,
            alpha=0.3
        ),
    )


ax[1].set_title(r"$\partial_y \phi$")
rel_err_dy = (100 * (result.se_dqvdy / np.abs(result.d_qv_dy))).sel(alt=slice(0, 1500)).groupby_bins("alt", bins=16).mean(dim="alt").to_pandas()

rel_err_dy.boxplot(
        ax=ax[1],
        vert=False,
        whis=(5, 95),
        patch_artist=True,
        boxprops=dict(
            facecolor="lightsteelblue",
            edgecolor="k"
        ),
        medianprops=dict(
            color="crimson",
            linewidth=2
        ),
        whiskerprops=dict(color="k"),
        capprops=dict(color="k"),
        flierprops=dict(
            marker="o",
            markersize=2,
            alpha=0.3
        ),
    )

for x in ax:
    x.set_xlim([0, 200])

In [ ]:
def fit2d_withErrs(x, y, u):
    """
    function for multilinear regression of u over x and y
    reference model :  phi = phi0 + dxU + dyU + res
    x, y = distances (m) from the circle centre
    u    = variable to regress
    
    xarray input shape: (..., altitude, sonde_id) — sonde_id is moved to the
    last axis by xr.apply_ufunc's input_core_dims, so all axis=-1 reductions
    here operate over sonde_id.
    computations are fully implemented in numpy

    to be called in fit2d_xr_loc()
    """

    # Convert everything to numpy
    x = np.asarray(x)
    y = np.asarray(y)
    u = np.array(u, copy=True)

    # Design matrix
    a = np.stack(
        [np.ones_like(x), x, y],
        axis=-1
    )

    # Invalid observations
    invalid = (
        np.isnan(u)
        | np.isnan(x)
        | np.isnan(y)
    )

    under_constraint = np.sum(~invalid, axis=-1) < 6

    # Remove invalid observations from fit
    u[invalid] = 0
    a[invalid] = 0

    # Pseudoinverse
    a_inv = np.linalg.pinv(a)

    # Regression coefficients
    intercept, dudx, dudy = np.einsum(
        "...rm,...m->r...",
        a_inv,
        u
    )

    # Covariance matrix factor
    # BE CAREFUL WITH THE INDICES AND DIMENSIONS HERE
    cov_beta = np.einsum("...ir,...jr->...ij", a_inv, a_inv)

    # Fitted values / residuals
    u_fit = (
        intercept[..., None]
        + dudx[..., None] * x
        + dudy[..., None] * y
    )

    u_err = u - u_fit

    # Don't let invalid points contribute to RSS
    u_err[invalid] = 0

    # Residual sum of squares
    u_sq_sum = np.sum(
        u_err**2,
        axis=-1
    )

    # Number of valid observations
    u_n = np.sum(
        ~invalid,
        axis=-1
    )

    # implement check :
    # the number of degrees of freedom should be more than 3
    dof = u_n - 3
    safe_dof = np.where(under_constraint, 1, dof)

    # Residual variance
    sigma2 = np.where(
        under_constraint,
        np.nan,
        u_sq_sum / safe_dof
    )

    # Standard errors
    se_beta = np.sqrt(
        sigma2[..., None]
        * np.diagonal(
            cov_beta,
            axis1=-2,
            axis2=-1
        )
    )

    se_intercept = se_beta[..., 0]
    se_dudx = se_beta[..., 1] 
    se_dudy = se_beta[..., 2] 

    # Enforce minimum number of observations
    intercept[under_constraint] = np.nan
    dudx[under_constraint] = np.nan
    dudy[under_constraint] = np.nan

    se_intercept[under_constraint] = np.nan
    se_dudx[under_constraint] = np.nan
    se_dudy[under_constraint] = np.nan

    return (
        intercept,
        dudx,
        dudy,
        se_intercept,
        se_dudx,
        se_dudy,
    )



def fit2d_xr_loc(x, y, u):
    """
    this function vectorizes fit2d_withErrs over an Xarray datarray
    """

    return xr.apply_ufunc(
        fit2d_withErrs,
        x,
        y,
        u,
        input_core_dims=[
            ["sonde_id"],
            ["sonde_id"],
            ["sonde_id"],
        ],
        output_core_dims=[
            [],
            [],
            [],
            [],
            [],
            [],
        ],
        vectorize=False,
    )




def fit_circle_loc(g, u_name):

    """ 
    this function allows to map the fit2d_xr_loc() on an xarray dataset, 
    selecting the particular variable with name u_name (str)

    since regressions should be carried out for every circle at each single altitude, 
    the xarray 
    """

    (
        intercept,
        dudx,
        dudy,
        se_intercept,
        se_dudx,
        se_dudy,
    ) = fit2d_xr_loc(
        g.x,
        g.y,
        g[u_name],
    )

    return xr.Dataset({
        f"{u_name}_mean": intercept,
        f"d_{u_name}_dx": dudx,
        f"d_{u_name}_dy": dudy,
        f"se_d{u_name}dx": se_dudx,
        f"se_d{u_name}dy": se_dudy,
        f"se_{u_name}_intercept": se_intercept,
    })


In [ ]:
joanne3_clean["qv"] = (joanne3_clean.q * joanne3_clean.v)

qv_fit = joanne3_clean.groupby("circle").map(lambda g: fit_circle_loc(g, "qv"))
qv_fit

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharey=True )
fig.suptitle("relative errors on gradients using general multivariate regression formulas")

ax[0].set_title(r"$\phi_0$")
rel_err_mean = (100 * (qv_fit.se_qv_intercept / np.abs(qv_fit.qv_mean))).sel(alt=slice(0, 1500)).groupby_bins("alt", bins=16).mean(dim="alt").to_pandas()

rel_err_mean.boxplot(
        ax=ax[0],
        vert=False,
        whis=(5, 95),
        patch_artist=True,
        boxprops=dict(
            facecolor="lightsteelblue",
            edgecolor="k"
        ),
        medianprops=dict(
            color="crimson",
            linewidth=2
        ),
        whiskerprops=dict(color="k"),
        capprops=dict(color="k"),
        flierprops=dict(
            marker="o",
            markersize=2,
            alpha=0.3
        ),
    )


ax[1].set_title(r"$\partial_x \phi$")
rel_err_dx = (100 * (qv_fit.se_dqvdx / np.abs(qv_fit.d_qv_dx))).sel(alt=slice(0, 1500)).groupby_bins("alt", bins=16).mean(dim="alt").to_pandas()

rel_err_dx.boxplot(
        ax=ax[1],
        vert=False,
        whis=(5, 95),
        patch_artist=True,
        boxprops=dict(
            facecolor="lightsteelblue",
            edgecolor="k"
        ),
        medianprops=dict(
            color="crimson",
            linewidth=2
        ),
        whiskerprops=dict(color="k"),
        capprops=dict(color="k"),
        flierprops=dict(
            marker="o",
            markersize=2,
            alpha=0.3
        ),
    )


ax[2].set_title(r"$\partial_y \phi$")
rel_err_dy = (100 * (qv_fit.se_dqvdy / np.abs(qv_fit.d_qv_dy))).sel(alt=slice(0, 1500)).groupby_bins("alt", bins=16).mean(dim="alt").to_pandas()

rel_err_dy.boxplot(
        ax=ax[2],
        vert=False,
        whis=(5, 95),
        patch_artist=True,
        boxprops=dict(
            facecolor="lightsteelblue",
            edgecolor="k"
        ),
        medianprops=dict(
            color="crimson",
            linewidth=2
        ),
        whiskerprops=dict(color="k"),
        capprops=dict(color="k"),
        flierprops=dict(
            marker="o",
            markersize=2,
            alpha=0.3
        ),
    )

for x in ax:
    x.set_xlim([0, 500])

In [ ]:
qv_prime = joanne3_clean.qv.groupby("circle") - qv_fit.qv_mean

In [ ]:
BCO_lon = -59.4288
BCO_lat = 13.1627

# - set control volume edges
vol_lon = [BCO_lon-0.5+360 , BCO_lon+15+360]  #[-60+360, -58+360]
vol_lat = [BCO_lat-5     , BCO_lat+5]  #[12.5, 14.5]


# - presenting area of interest

fig = plt.figure(figsize=(13, 7))
ax = plt.axes(projection=ccrs.PlateCarree())

# Add base features
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set_extent([vol_lon[0] - 1., vol_lon[1] + 30., vol_lat[0] - 8., vol_lat[1] + 5.], crs=ccrs.PlateCarree())


gl = ax.gridlines(draw_labels=True, 
                  linewidth=0.5, color='gray', alpha=0.5, linestyle='--')

# Turn on labels on bottom and left only
gl.top_labels = False
gl.right_labels = False

# Optionally format ticks (degrees + symbols)
gl.xformatter = cticker.LongitudeFormatter()
gl.yformatter = cticker.LatitudeFormatter()




## scatter the dropsondes lon and lat
sc = ax.scatter(joanne3_clean.lon.isel(alt=2), joanne3_clean.lat.isel(alt=2), c=qv_prime.sel(alt=slice(50, 500)).mean(dim="alt"),
                 cmap="seismic", vmin=-0.15, vmax=0.15,
                 transform=ccrs.PlateCarree())

cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label(r"$\rm m~s^{-1}~kg~kg^{-1}$")
ax.set_title(r"$qv$ in each single dropsonde")


In [ ]:
import dropsonde_fit_library

In [ ]:
q_prime = joanne3_clean.q.groupby("circle") - joanne4.q
v_prime = joanne3_clean.v.groupby("circle") - joanne4.v

joanne3_clean["q_prime_v_prime"] = q_prime * v_prime
qp_vp_fit = joanne3_clean.groupby("circle").map(lambda g: dropsonde_fit_library.fit_circle_loc(g, "q_prime_v_prime"))
qp_vp_fit

In [ ]:
qv_fit = qv_fit.assign_coords(circle_lat=("circle", joanne4.circle_lat.values)) 
qp_vp_fit = qp_vp_fit.assign_coords(circle_lat=("circle", joanne4.circle_lat.values))

qv_fit = qv_fit.assign_coords(circle_lon=("circle", joanne4.circle_lon.values)) 
qp_vp_fit = qp_vp_fit.assign_coords(circle_lon=("circle", joanne4.circle_lon.values))

In [ ]:
qb_vb = qv_fit.qv_mean - qp_vp_fit.q_prime_v_prime_mean
sigma_qb_vb = (qv_fit.se_qv_intercept**2 + qp_vp_fit.se_q_prime_v_prime_intercept**2)**0.5

In [ ]:
subcloud_layer = slice(0,500)

## qv bar
qv_b_err = np.sqrt((qv_fit.se_qv_intercept**2).sel(alt=subcloud_layer).sum(dim="alt", skipna=True) / (qv_fit.se_qv_intercept).sel(alt=subcloud_layer).notnull().sum(dim="alt"))
y0 = qv_fit.qv_mean.sel(alt=subcloud_layer).mean(dim="alt").sortby("circle_lat")

fitw = y0.swap_dims({"circle": "circle_lat"}).polyfit(dim="circle_lat", deg=1, w=(1/qv_b_err)**2, cov=True)
bw = fitw["polyfit_coefficients"].sel(degree=1)  # slope
aw = fitw["polyfit_coefficients"].sel(degree=0)  # intercept

bw_err = np.sqrt(fitw["polyfit_covariance"].sel(cov_i=1, cov_j=1))


# standard errors from covariance matrix
# cov = fitw["polyfit_covariance"]
# se_b = np.sqrt(cov.sel(degree=1, cov_degree=1))
# se_a = np.sqrt(cov.sel(degree=0, cov_degree=0))


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
titles = [r"$\overline{qv}$", r"$\overline{q'v'}$", r"$ \overline{qv} - \overline{q'v'} = \overline{q}~\overline{v} $"]


## qv bar
ax[0].errorbar(y0.circle_lat, y0, yerr=qv_b_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")
ax[0].plot( y0.circle_lat, aw + bw*y0.circle_lat, color="orange", alpha=0.4, label=f"conv. WEST = {(-86400*bw.values/(111)):.2f} $\pm$ {(bw_err.values*(86400/111)):.2f} $g~kg^{-1}~d^{-1}$"  )


ax[0].legend()
ax[0].set_ylabel(r"$\rm kg~kg^{-1} ~~ m~s^{-1}$")


## (q prime v prime) bar
qp_vp_bar_err = np.sqrt((qp_vp_fit.se_q_prime_v_prime_intercept**2).sel(alt=subcloud_layer).sum(dim="alt", skipna=True) / (qp_vp_fit.se_q_prime_v_prime_intercept).sel(alt=subcloud_layer).notnull().sum(dim="alt"))
y1 = qp_vp_fit.q_prime_v_prime_mean.sel(alt=subcloud_layer).mean(dim="alt").sortby("circle_lat")

ax[1].errorbar(y1.circle_lat, y1, yerr=qp_vp_bar_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")


## q bar v bar
y = y0 - y1
err_y = np.sqrt(qv_b_err**2 + qp_vp_bar_err**2)

ax[2].errorbar( y.circle_lat, y, yerr=err_y.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")


## esthetics
for i, x in enumerate(ax):
    x.set_title(titles[i])
    x.set_xlabel(r"lat / $\degree$")

    for spine in ["top", "right"]:
        x.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        x.spines[spine].set_position(("outward", 10))

especially on the WEST, the zonl transport should be alo important

In [ ]:
joanne3_clean["qu"] = (joanne3_clean.q * joanne3_clean.u)

qu_fit = joanne3_clean.groupby("circle").map(lambda g: fit_circle_loc(g, "qu"))


# compute primes
q_prime = joanne3_clean.q.groupby("circle") - joanne4.q
u_prime = joanne3_clean.u.groupby("circle") - joanne4.u

joanne3_clean["q_prime_u_prime"] = q_prime * u_prime
qp_up_fit = joanne3_clean.groupby("circle").map(lambda g: dropsonde_fit_library.fit_circle_loc(g, "q_prime_u_prime"))


# assign circle coordiantes
qu_fit = qu_fit.assign_coords(circle_lat=("circle", joanne4.circle_lat.values)) 
qp_up_fit = qp_up_fit.assign_coords(circle_lat=("circle", joanne4.circle_lat.values))

qu_fit = qu_fit.assign_coords(circle_lon=("circle", joanne4.circle_lon.values)) 
qp_up_fit = qp_up_fit.assign_coords(circle_lon=("circle", joanne4.circle_lon.values))


# compute large scale only contributions
qb_ub = qu_fit.qu_mean - qp_up_fit.q_prime_u_prime_mean
sigma_qb_ub = (qu_fit.se_qu_intercept**2 + qp_up_fit.se_q_prime_u_prime_intercept**2)**0.5



subcloud_layer = slice(0,500)

# ## qv bar
qu_b_err = np.sqrt((qu_fit.se_qu_intercept**2).sel(alt=subcloud_layer).sum(dim="alt", skipna=True) / (qu_fit.se_qu_intercept).sel(alt=subcloud_layer).notnull().sum(dim="alt"))
y0 = qu_fit.qu_mean.sel(alt=subcloud_layer).mean(dim="alt").sortby("circle_lat")

# fitw = y0.where(west, drop=True).swap_dims({"circle": "circle_lat"}).polyfit(dim="circle_lat", deg=1, w=(1/qv_b_err.where(west, drop=True))**2, cov=True)
# bw = fitw["polyfit_coefficients"].sel(degree=1)  # slope
# aw = fitw["polyfit_coefficients"].sel(degree=0)  # intercept

# fite = y0.where(east, drop=True).swap_dims({"circle": "circle_lat"}).polyfit(dim="circle_lat", deg=1, w=(1/qv_b_err.where(east, drop=True))**2, cov=True)
# be = fite["polyfit_coefficients"].sel(degree=1)  # slope
# ae = fite["polyfit_coefficients"].sel(degree=0)  # intercept

# standard errors from covariance matrix
# cov = fitw["polyfit_covariance"]
# se_b = np.sqrt(cov.sel(degree=1, cov_degree=1))
# se_a = np.sqrt(cov.sel(degree=0, cov_degree=0))


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
titles = [r"$\overline{qu}$", r"$\overline{q'u'}$", r"$ \overline{qu} - \overline{q'u'} = \overline{q}~\overline{u} $"]


## qv bar
ax[0].errorbar(y0.circle_lat, y0, yerr=qu_b_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")
# ax[0].plot( y0.circle_lat, aw + bw*y0.circle_lat, color="orange", alpha=0.4, label=f"conv. WEST = {(-86400*bw.values/(111)):.2f} $g~kg^{-1}~d^{-1}$"  )


ax[0].legend()
ax[0].set_ylabel(r"$\rm kg~kg^{-1} ~~ m~s^{-1}$")


## (q prime v prime) bar
qp_up_bar_err = np.sqrt((qp_up_fit.se_q_prime_u_prime_intercept**2).sel(alt=subcloud_layer).sum(dim="alt", skipna=True) / (qp_up_fit.se_q_prime_u_prime_intercept).sel(alt=subcloud_layer).notnull().sum(dim="alt"))
y1 = qp_up_fit.q_prime_u_prime_mean.sel(alt=subcloud_layer).mean(dim="alt").sortby("circle_lat")

ax[1].errorbar(y1.circle_lat, y1, yerr=qp_up_bar_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")


## q bar v bar
y = y0 - y1
err_y = np.sqrt(qu_b_err**2 + qp_up_bar_err**2)

ax[2].errorbar( y.circle_lat, y, yerr=err_y.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")



## esthetics
for i, x in enumerate(ax):
    x.set_title(titles[i])
    x.set_xlabel(r"lat / $\degree$")

    for spine in ["top", "right"]:
        x.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        x.spines[spine].set_position(("outward", 10))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4), constrained_layout=True)
titles = [r"$\overline{qu}$"]


## qv bar
ax.errorbar(y0.circle_lon, y0, yerr=qu_b_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")


ax.errorbar(y0.circle_lon, y0, yerr=qu_b_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="blue")

ax.legend()
ax.set_ylabel(r"$\rm kg~kg^{-1} ~~ m~s^{-1}$")


ax.set_title(titles[0])
ax.set_xlabel(r"lon / $\degree$")

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_position(("outward", 10))

#### same, but for gradients now

meridional

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)

subcloud_layer = slice(0,500)

titles = [r"$-\partial_y \overline{qv}$", r"$-\partial_y \overline{q'v'}$", r"$-\partial_y \overline{qv} + \partial_y \overline{q'v'} = -\partial_y\overline{q}~\overline{v} $"]


## qv bar
qv_b_err = np.sqrt((qv_fit.se_dqvdy**2).sel(alt=subcloud_layer).sum(dim="alt", skipna=True) / (qv_fit.se_dqvdy).sel(alt=subcloud_layer).notnull().sum(dim="alt"))
y0 = -qv_fit.d_qv_dy.sel(alt=subcloud_layer).mean(dim="alt").sortby("circle_lat")

ax[0].errorbar(y0.circle_lat, y0, yerr=qv_b_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")

ax[0].set_ylabel(r"$\rm kg~kg^{-1} ~~ s^{-1}$")


## (q prime v prime) bar
qp_vp_bar_err = np.sqrt((qp_vp_fit.se_dq_prime_v_primedy**2).sel(alt=subcloud_layer).sum(dim="alt", skipna=True) / (qp_vp_fit.se_dq_prime_v_primedy).sel(alt=subcloud_layer).notnull().sum(dim="alt"))
y1 = -qp_vp_fit.d_q_prime_v_prime_dy.sel(alt=subcloud_layer).mean(dim="alt").sortby("circle_lat")

ax[1].errorbar(y1.circle_lat, y1, yerr=qp_vp_bar_err.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")



## q bar v bar
y = (y0 - y1)
err_y = np.sqrt(qv_b_err**2 + qp_vp_bar_err**2)

ax[2].errorbar( y.circle_lat, y, yerr=err_y.sortby("circle_lat"),
    fmt="o",
    markersize=2.5,
    linestyle="none",
    color="orange")




## esthetics
for i, x in enumerate(ax):
    x.set_title(titles[i])
    x.set_xlabel(r"lat / $\degree$")

    for spine in ["top", "right"]:
        x.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        x.spines[spine].set_position(("outward", 10))

#### now let's lok at the sole advective terms:
$$\overline{v}~\partial_y \overline{q}$$
the error on this term is 
$$ 
    \sqrt{ (v~\sigma_{\partial})^2 + (\sigma_v ~ \partial)^2 }
$$

In [ ]:
joanne4

In [ ]:
v_fit = joanne3_clean.groupby("circle").map(lambda g: dropsonde_fit_library.fit_circle_loc(g, "v"))

# we already have the area-averaged gradient of moisture

vb_dqb = v_fit.v_mean * joanne4.dqdy

vb_dqb_err = np.sqrt( ( v_fit.v_mean*joanne4.se_dqdy )**2 + ( v_fit.se_v_intercept*joanne4.dqdy )**2  )

now we need the primes statistics, averaged over the circles <br>
$$\overline{v'~\partial_y q'}$$

In [ ]:
joanne3_clean["qp"] = joanne3_clean.q.groupby("circle") - joanne4.q
vp = joanne3_clean.v.groupby("circle") - joanne4.v

### testbed for polar coordinates

In [ ]:
z = joanne3_clean.qp.isel(alt=100).isel(sonde_id=slice(0, 10))
xx, yy = joanne3_clean.x.isel(alt=100).isel(sonde_id=slice(0, 10)), joanne3_clean.y.isel(alt=100).isel(sonde_id=slice(0, 10))

plt.scatter( xx, yy, c=z  )

In [ ]:
"""## this is the routing for a single circle

r = np.sqrt(x**2 + y**2)
theta = np.arctan2(y, x)
theta_unwrapped = np.unwrap(theta)

s = r * theta_unwrapped

z = z.assign_coords(arc=("sonde_id", s.values if hasattr(s, "values") else s))

dz_ds = z.differentiate("arc")
dz_ds_vals = dz_ds.values          # shape (n,)

theta_vals = theta.values if hasattr(theta, "values") else theta  # shape (n,)
n = theta_vals.shape[0]

cos_t = np.cos(theta_vals)
sin_t = np.sin(theta_vals)

grad_x = -sin_t * dz_ds
grad_y =  cos_t * dz_ds"""

In [ ]:
r = np.sqrt(joanne3_clean.x**2 + joanne3_clean.y**2)
theta = np.arctan2(joanne3_clean.y, joanne3_clean.x)

def unwrap_along_sonde(g):
    return xr.apply_ufunc(
        np.unwrap, g,
        input_core_dims=[["sonde_id"]],
        output_core_dims=[["sonde_id"]],
        keep_attrs=True,
    )

theta_unwrapped = theta.groupby("circle").map(unwrap_along_sonde)

s = r * theta_unwrapped
s
joanne3_clean = joanne3_clean.assign_coords(arc=(("sonde_id", "alt"),s.values))
joanne3_clean

In [ ]:
def differentiate_qp(group):
    qp = group["qp"].values
    arc = group["arc"].values

    n_sondes, n_altitudes = qp.shape

    result = np.full_like(qp, np.nan, dtype=np.result_type(qp, float))

    # Interior points: central difference
    result[1:-1, :] = (
        (qp[2:, :] - qp[:-2, :]) / (arc[2:, :] - arc[:-2, :])
    )

    # First sonde: forward difference
    result[0, :] = (
        (qp[1, :] - qp[0, :])
        / (arc[1, :] - arc[0, :])
    )

    # Last sonde: backward difference
    result[-1, :] = (
        (qp[-1, :] - qp[-2, :])
        / (arc[-1, :] - arc[-2, :])
    )

    return xr.DataArray(
        result,
        dims=group["qp"].dims,
        coords=group["qp"].coords,
        name="dqp_darc",
    )


dqp_darc = joanne3_clean.groupby("circle").map(differentiate_qp)

cos_t = np.cos(theta)
sin_t = np.sin(theta)

dqp_x = -sin_t * dqp_darc
dqp_y =  cos_t * dqp_darc

In [ ]:
joanne3_clean["vp_dqp_dy"] = vp * dqp_y
vp_dqp_dy_fit = joanne3_clean.groupby("circle").map(lambda g: dropsonde_fit_library.fit_circle_loc(g, "vp_dqp_dy"))

vp_dqp_dy_fit = vp_dqp_dy_fit.assign_coords(circle_lat=("circle", joanne4.circle_lat.values), circle_time=("circle", joanne4.circle_time.values))
vp_dqp_dy_fit = vp_dqp_dy_fit.assign_coords(circle_lon=("circle", joanne4.circle_lon.values))

vp_dqp_dy_fit

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
fig.suptitle("EUREC4A circles only")

low_layer = slice(0,1500)
gkg_perday = 1000*86400
perday     = 86400

y0 = gkg_perday * (-vb_dqb.sel(alt=low_layer).sortby("circle_time"))

p1 = y0.plot(
    ax=ax[0], x="circle_time", y="alt", vmin=-2.5, vmax=2.5, cmap="coolwarm",
)


y1 = perday * (-vp_dqp_dy_fit.vp_dqp_dy_mean.sel(alt=low_layer).sortby("circle_time"))

p3 = y1.plot(
    ax=ax[1], x="circle_time", y="alt", vmin=-2.5, vmax=2.5, cmap="coolwarm",
)


## relative errors on v' dq'
y2 = ((100*np.abs(vp_dqp_dy_fit.vp_dqp_dy_mean) / vp_dqp_dy_fit.se_vp_dqp_dy_intercept).sel(alt=low_layer).sortby("circle_time"))

p2 = y2.plot(
    ax=ax[2], x="circle_time", y="alt", vmin=0, vmax=200, cmap="viridis",
)


for p in ( p1, p3, p2):
    p.colorbar.set_label(r"$\rm g~kg~day^{-1}$")

titles = [ r"$-\overline{v} \partial_y \overline{q}$", r"$-\overline{v'\partial_y q'}$", r"relative errors on $-\overline{v'\partial_y q'}$"]

for i, x in enumerate(ax.flatten()):
    x.set_title(titles[i])
    x.set_xlabel(r"lat / $\degree$")

    for spine in ["top", "right"]:
        x.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        x.spines[spine].set_position(("outward", 10))